# EcoPulse AI — Exploratory Data Analysis

UCI Appliances Energy Prediction Dataset  
https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction

This notebook provides an interactive exploration of the dataset used to train the EcoPulse AI models.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data_loader import load_raw_data
from src.preprocessing import clean_data, chronological_split
from src.feature_engineering import prepare_features

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.family'] = 'sans-serif'

print('Libraries loaded successfully!')

In [ ]:
# Load the raw dataset
df = load_raw_data()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Dataset info
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# Target variable distribution
print('=== Appliances Energy Summary ===')
print(df['Appliances'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['Appliances'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Appliance Energy (Wh)')
axes[0].set_xlabel('Appliances (Wh)')
axes[0].set_ylabel('Frequency')

df['Appliances'].plot(kind='box', ax=axes[1], color='steelblue')
axes[1].set_title('Box Plot: Appliance Energy')
axes[1].set_ylabel('Wh')
plt.tight_layout()
plt.show()

In [ ]:
# Clean data and create time features
cleaned = clean_data(df)
cleaned['date'] = pd.to_datetime(cleaned['date'])

# Energy consumption over time
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(cleaned['date'], cleaned['Appliances'], linewidth=0.6, color='#2E7D32', alpha=0.8)
ax.set_title('Appliance Energy Consumption Over Time', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Energy (Wh)')
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Hourly patterns
cleaned['hour'] = cleaned['date'].dt.hour
hourly_avg = cleaned.groupby('hour')['Appliances'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
hourly_avg.plot(kind='bar', ax=ax, color='#00796B', edgecolor='white')
ax.set_title('Average Energy Consumption by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Average Energy (Wh)')
ax.set_xticklabels([str(h) for h in range(24)], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
numeric_cols = cleaned.select_dtypes(include=np.number).columns
corr = cleaned[numeric_cols].corr()['Appliances'].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
corr.drop('Appliances').plot(kind='barh', ax=ax, color=['#2E7D32' if v > 0 else '#C62828' for v in corr.drop('Appliances')])
ax.set_title('Feature Correlation with Appliance Energy')
ax.set_xlabel('Pearson Correlation')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Weekend vs weekday comparison
cleaned['is_weekend'] = (cleaned['date'].dt.dayofweek >= 5).astype(int)
cleaned['day_type'] = cleaned['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})

fig, ax = plt.subplots(figsize=(8, 4))
cleaned.boxplot(column='Appliances', by='day_type', ax=ax, 
                boxprops=dict(color='steelblue'),
                medianprops=dict(color='red', linewidth=2))
ax.set_title('Energy Consumption: Weekday vs Weekend')
ax.set_xlabel('')
ax.set_ylabel('Energy (Wh)')
plt.suptitle('')
plt.tight_layout()
plt.show()

print(cleaned.groupby('day_type')['Appliances'].describe())

In [ ]:
# Train/test split visualization
train_df, test_df = chronological_split(cleaned)

print(f'Train: {train_df.shape[0]} rows | {train_df["date"].min()} → {train_df["date"].max()}')
print(f'Test:  {test_df.shape[0]} rows | {test_df["date"].min()} → {test_df["date"].max()}')

X_train, y_train = prepare_features(train_df)
X_test, y_test = prepare_features(test_df)
print(f'\nFeatures ({len(X_train.columns)}): {list(X_train.columns)}')